# Validate on held-out sequences
Greedy generation; no gradients or training. P IDs are evaluated by their appearance groups, not their arbitrary numbering.

Choose the saved run.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "project.json").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pipeline_helpers as P
import json
from PIL import Image, ImageDraw, ImageFont
ROOT = P.ROOT
MODELS = P.read_json(ROOT / "models.json")
SPLITS = ["train", "val"]  # Or ["train"] / ["val"].
FORCE_RERUN = False
items = P.pages(SPLITS)
print("Pages:", len(items))
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
from weighted_tokens_helpers import load_joint_sample
from evaluation_helpers import metrics
CHECKPOINT = ROOT / "data/training/runs/weighted_tokens_full_assistant_all_train_01"
SAMPLE_PATHS = P.exported_samples("val")
samples = [load_joint_sample(path) for path in SAMPLE_PATHS]
run_config = P.read_json(CHECKPOINT / "config.json")
if run_config["split_manifest_sha256"] != P.file_hash(P.MANIFEST):
    raise ValueError("Checkpoint and validation split differ.")
OUTPUT = CHECKPOINT / "validation.json"
if OUTPUT.exists():
    raise FileExistsError("Choose a new OUTPUT path to preserve prior results.")


Restore the local base and adapter.

In [ ]:
import torch
H.cleanup()
model, tokenizer = H.load_base()
config = H.restore_adapters(model, CHECKPOINT)
model.eval()


Save each prediction and aggregate page/bubble metrics.

In [ ]:
results = []
for path, sample in zip(SAMPLE_PATHS, samples):
    pixels = H.tiles(sample, config["max_tiles"], view="annotated")
    pixels = pixels.to(device="cuda", dtype=next(model.vision_model.parameters()).dtype)
    with torch.inference_mode():
        response = model.chat(tokenizer, pixels, sample["prompt"], {"max_new_tokens":1024, "do_sample":False})
    row = {"sample_path":str(path), "page_id":sample["page_id"], "sequence_id":sample["group_id"],
           "page_number":P.read_json(path)["page_number"], "split":"val", "response":response,
           "target":sample["target"], "valid_json":False, "valid_schema":False}
    try:
        prediction = json.loads(response)
        row.update(prediction=prediction, valid_json=True)
        row.update(metrics(prediction, sample["target"]), valid_schema=True)
    except (ValueError, TypeError, KeyError) as error:
        row["error"] = str(error)
    results.append(row)
    P.write_json(OUTPUT, {"results":results})
    print(sample["page_id"], row["valid_schema"])
    del pixels
total_bubbles = sum(len(row["target"]["bubble_speakers"]) for row in results)
unknown_total = sum(sum(s == "UNKNOWN" for s in row["target"]["bubble_speakers"].values()) for row in results)
summary = {"pages":len(results), "valid_json":sum(r["valid_json"] for r in results),
    "valid_schema":sum(r["valid_schema"] for r in results),
    "exact_semantic_pages":sum(r.get("exact_semantic_match", False) for r in results),
    "grouping_correct_pages":sum(r.get("grouping_correct", False) for r in results),
    "rejection_correct_pages":sum(r.get("rejection_correct", False) for r in results),
    "speaker_accuracy":sum(r.get("speaker_correct", 0) for r in results) / total_bubbles if total_bubbles else None,
    "unknown_accuracy":sum(r.get("unknown_correct", 0) for r in results) / unknown_total if unknown_total else None}
P.write_json(OUTPUT, {"summary":summary, "results":results})
print(summary)


## View saved results
These cells only read saved results. They do not run models or change cached files. Training shows targets and run-wide loss history; validation shows saved predictions.

In [ ]:
from pathlib import Path
import sys, importlib
view_root = Path.cwd().resolve()
if not (view_root / "project.json").is_file():
    view_root = view_root.parent
sys.path.insert(0, str(view_root))
import result_viewer as V
importlib.reload(V)
VIEW_STAGE = 'validation'
VIEW_SPLIT = "val"
VIEW_RUN = Path(CHECKPOINT) if "CHECKPOINT" in globals() else view_root / "data/training/runs/weighted_tokens_full_assistant_all_train_01"


### 1. View a random page
Rerun this cell to choose another page with a saved result in this split.

In [ ]:
viewed_page = V.show_random(VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)


### 2. View all pages of a sequence
Enter a sequence ID, or leave it empty to use the random page’s sequence. Missing stage results are reported for each page.

In [ ]:
VIEW_SEQUENCE_ID = ""  # Example: "seq_71033fec4de494fc"
sequence_to_view = VIEW_SEQUENCE_ID or (viewed_page["sequence_id"] if viewed_page else "")
if sequence_to_view:
    V.show_sequence(sequence_to_view, VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)
else:
    print("Set VIEW_SEQUENCE_ID to a sequence in VIEW_SPLIT.")
